# Курсовая работа
Студент: Виноградова Валерия Михайловна

Группа: ИД24-3

Тема 7: Анализ характеристик музыкальных треков для поиска паттернов
популярности и жанровой принадлежности.

## 2.2. Подбор источников данных и выбор характеристик


In [26]:
import pandas as pd
import numpy as np
import time
import os

### 1. Реляционный набор данных CSV (Kaggle)
Содержит метаданные, аудио-характеристики и показатели популярности треков. 

In [20]:
import pandas as pd

spotify_df = pd.read_csv('spotify_tracks.csv')

print(spotify_df.shape)
display(spotify_df.head(3))
print(spotify_df.isnull().sum().sum())

(62317, 22)


,track_id,track_name,artist_name,year,popularity,artwork_url,album_name,acousticness,danceability,duration_ms,...,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence,track_url,language
0,2r0ROhr7pRN4MXDMT1fEmd,"Leo Das Entry (From ""Leo"")",Anirudh Ravichander,2024,59,https://i.scdn.co/image/ab67616d0000b273ce9c65...,"Leo Das Entry (From ""Leo"")",0.0241,0.753,97297.0,...,8.0,0.1000,-5.994,0.0,0.1030,110.997,4.0,0.459,https://open.spotify.com/track/2r0ROhr7pRN4MXD...,Tamil
1,4I38e6Dg52a2o2a8i5Q5PW,AAO KILLELLE,"Anirudh Ravichander, Pravin Mani, Vaishali Sri...",2024,47,https://i.scdn.co/image/ab67616d0000b273be1b03...,AAO KILLELLE,0.0851,0.780,207369.0,...,10.0,0.0951,-5.674,0.0,0.0952,164.995,3.0,0.821,https://open.spotify.com/track/4I38e6Dg52a2o2a...,Tamil
2,59NoiRhnom3lTeRFaBzOev,Mayakiriye Sirikiriye - Orchestral EDM,"Anirudh Ravichander, Anivee, Alvin Bruno",2024,35,https://i.scdn.co/image/ab67616d0000b27334a1dd...,Mayakiriye Sirikiriye (Orchestral EDM),0.0311,0.457,82551.0,...,2.0,0.0831,-8.937,0.0,0.1530,169.996,4.0,0.598,https://open.spotify.com/track/59NoiRhnom3lTeR...,Tamil


0



### 2. Источник XML (MusicBrainz)

Структура файла представлена в виде дерева: корневой тег `release` содержит вложенные теги `title`, `date` и `artist`. 

In [25]:
import xml.etree.ElementTree as ET
import pandas as pd

tree = ET.parse('musicbrainz_real.xml')
root = tree.getroot()

data = []
for r in root.findall('.//{http://musicbrainz.org/ns/mmd-2.0#}release')[:10]:
    data.append({
        'title': r.findtext('{http://musicbrainz.org/ns/mmd-2.0#}title'),
        'date': r.findtext('{http://musicbrainz.org/ns/mmd-2.0#}date'),
        'artist': r.findtext('.//{http://musicbrainz.org/ns/mmd-2.0#}name')
    })

display(pd.DataFrame(data))

,title,date,artist
0,Roke Na Ruke Naina,2023-03-22,Smoothest Aashu
1,I’m Jus’ Fuckin’ With U Gurl,2023-03-12,Thuggizzle
2,Fuck Xmas!,2023,Spiderlegs
3,Diamond Star Collection,2023-03-22,"Dave Dee, Dozy, Beaky, Mick & Tich"
4,Live at South Yarmouth,2023,Aerosmith
5,¿Para qué?,2023-11-17,Hanna Rivas
6,La Danse Macabre,2023,Memento Mori
7,明日につながる道,2023-04-01,渡辺学
8,"(picnic, lightning)",2023-03-27,Garlands
9,Live at Acton Town Hall,2023-08,Joe Strummer & the Mescaleros


### 3. Источник с JSON (AcousticBrainz)
Содержит результаты акустического анализа треков (BPM, танцевальность, тональность).

In [21]:
import json
import glob
import pandas as pd

ab_data = []

for file_path in glob.glob('ab_data/**/*.json', recursive=True)[:5]:
    with open(file_path, 'r') as f:
        data = json.load(f)
        
        ab_data.append({
            'bpm': data.get('rhythm', {}).get('bpm'),
            'danceability': data.get('rhythm', {}).get('danceability'),
            'loudness': data.get('lowlevel', {}).get('average_loudness')
        })
        
ab_df = pd.DataFrame(ab_data)
display(ab_df)

,bpm,danceability,loudness
0,94.319115,0.879877,0.150803
1,93.120735,1.241049,0.827158
2,155.999847,1.079882,0.810807
3,97.000565,1.347988,0.955147
4,93.049492,1.172791,0.841683


## 4. Неструктурированный источник данных База данных SQLite (Kaggle)
В качестве источника неструктурированных текстовых данных был выбран датасет музыкальных рецензий Pitchfork Reviews Dataset. Он содержит тексты обзоров альбомов и оценки критиков. 

In [15]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('database.sqlite')

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Таблицы в базе данных:")
display(tables)

Таблицы в базе данных:


,name
0,reviews
1,artists
2,genres
3,labels
4,years
5,content


### Итог 2.2

Собраны 4 источника данных. Выделены признаки для дальнейшего анализа:

Числовые:
- popularity (целевая переменная, Spotify)
- danceability, energy, valence, tempo, loudness (Spotify)
- bpm, average_loudness (AcousticBrainz)
- rating (Pitchfork)

Категориальные:
- track_genre (Spotify)
- artists (Spotify)
- key (AcousticBrainz)
- track_name, artist (MusicBrainz)

Временные:
- year (MusicBrainz)